# **Durability emulator — dataset generation**

This notebook **only** runs the emulator and saves the resulting lambda datasets — no PCE is fitted
here. [`02_train_pce.ipynb`](02_train_pce.ipynb) only reads what this notebook writes.

The simulator is the closed-form carbonation model of Possan et al. (2016),
`carbonation_depth_possan_by_type` — no trained ML model is loaded.

**Artefacts written per time step**,
`<n_latent_samples>_<kind>_<split>_<t>_install_<year>_cement_<type>_exposure_<exposure>.pkl`:

| kind | split | content |
|---|---|---|
| `dataset_full` | train / val | one row per latent replica: inputs, effective values, carbonation depth, $g$, lambdas, processing time |
| `dataset_unique` | train / val | one row per design point: inputs, the four lambdas, processing time |

Plus one aggregate `<n_latent_samples>_emulator_timing_durability.pkl`, read back by
[`02_train_pce.ipynb`](02_train_pce.ipynb) to compute the emulator/surrogate speed-up.

CO₂ uses the published CMIP6/SSP table. Set `installation_year` and `co2_scenario` below.
The 100-year horizon must end no later than 2100. Regenerate datasets and retrain after changing the scenario; old polynomial results are not compatible.


## **1. Libraries**

In [2]:
import sys
import time
from pathlib import Path

# functions.py sits one directory up
sys.path.insert(0, str(Path.cwd().parent))

import dill
import numpy as np
import pandas as pd

from functions import *
from UQpy.distributions import Uniform, JointIndependent

c:\git-projetos\2024-1_victor_hugo_renata_maria\.venv\Lib\site-packages\UQpy\__init__.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


## **2. Random variables and fixed parameters**

Design variables (compressive strength, relative humidity, cover) plus everything the emulator
needs that isn't a design variable.

In [3]:
fck_min = 20  # MPa
fck_max = 50  # MPa
rh_min  = 20  # %
rh_max  = 80  # %
cov_min = 15  # mm
cov_max = 60  # mm

cement_type          = 3
installation_year    = 1980
co2_scenario         = "SSP2-4.5"  # SSP1-2.6, SSP2-4.5, or SSP5-8.5
exposure_conditions  = 2
ad                   = 10.0      # Pozzolanic material in the concrete, relative to cement mass (%) — Possan et al. (2016) model input
n_samples            = 200      # Number of design samples
n_latent_samples     = 2500     # Number of latent samples per design sample. Also the filename prefix
n_samples_validation = 50       # Number of validation samples, redrawn at every time step
n_lambdas            = 4        # Number of λs (λ1, λ2, λ3, λ4)

## **3. Design samples**

In [4]:
fck_dist = Uniform(loc=fck_min, scale=fck_max - fck_min)
rh_dist  = Uniform(loc=rh_min, scale=rh_max - rh_min)
cov_dist = Uniform(loc=cov_min, scale=cov_max - cov_min)
joint    = JointIndependent(marginals=[fck_dist, rh_dist, cov_dist])

x_pce_rvs = joint.rvs(n_samples)
x_val = joint.rvs(n_samples_validation)

print("Samples generated successfully!")
print(f"   Number of design samples: {n_samples}")
print(f"   Number of latent samples per design sample: {n_latent_samples}")
print(f"   Total simulations per time step: {(n_samples + n_samples_validation) * n_latent_samples}")
print("\nSample statistics:")
print(f"   fck:   {x_pce_rvs[:, 0].min():.1f} - {x_pce_rvs[:, 0].max():.1f} MPa (mean: {x_pce_rvs[:, 0].mean():.1f} MPa)")
print(f"   RH:    {x_pce_rvs[:, 1].min():.1f} - {x_pce_rvs[:, 1].max():.1f}% (mean: {x_pce_rvs[:, 1].mean():.1f}%)")
print(f"   cover: {x_pce_rvs[:, 2].min():.1f} - {x_pce_rvs[:, 2].max():.1f} mm (mean: {x_pce_rvs[:, 2].mean():.1f} mm)")

Samples generated successfully!
   Number of design samples: 200
   Number of latent samples per design sample: 2500
   Total simulations per time step: 625000

Sample statistics:
   fck:   20.0 - 49.8 MPa (mean: 34.2 MPa)
   RH:    20.0 - 79.8% (mean: 48.4%)
   cover: 15.2 - 59.7 mm (mean: 37.2 mm)


## **4. Time grid**

In [5]:
times = np.linspace(0, 100, 5, endpoint=True)  # Time points for carbonation depth prediction
times

array([  0.,  25.,  50.,  75., 100.])

## **5. Generate the dataset at each time step**

The same `x_val` (drawn once, in section 3) is reused across every time step — so the same
validation design points can be tracked over time, instead of being redrawn independently at each
`t`. `generate_dataset_at_time_durability` does the rest: latent sampling, carbonation depth
prediction, GLD fit, and saving `dataset_full`/`dataset_unique` for both splits.

In [6]:
print("="*60)
print("GENERATING THE DURABILITY DATASET")
print("="*60)

generation_results = []
for t in times:
    result = generate_dataset_at_time_durability(
                                                    x_train=x_pce_rvs,
                                                    x_val=x_val,
                                                    time_step=t,
                                                    cement_type=cement_type,
                                                    installation_year=installation_year,
                                                    co2_scenario=co2_scenario,
                                                    exposure_conditions=exposure_conditions,
                                                    ad=ad,
                                                    n_latent_samples=n_latent_samples,
                                                    output_dir='.',
                                                )
    generation_results.append(result)

GENERATING THE DURABILITY DATASET

----------------------------------------
GENERATING DATASET FOR TIME STEP: 0.0 years
----------------------------------------
  train: 200 design points, 28.88 s total
  val: 50 design points, 7.41 s total

----------------------------------------
GENERATING DATASET FOR TIME STEP: 25.0 years
----------------------------------------
  train: 200 design points, 20.70 s total
  val: 50 design points, 8.29 s total

----------------------------------------
GENERATING DATASET FOR TIME STEP: 50.0 years
----------------------------------------
  train: 200 design points, 36.24 s total
  val: 50 design points, 10.79 s total

----------------------------------------
GENERATING DATASET FOR TIME STEP: 75.0 years
----------------------------------------
  train: 200 design points, 41.83 s total
  val: 50 design points, 7.44 s total

----------------------------------------
GENERATING DATASET FOR TIME STEP: 100.0 years
----------------------------------------
  tra

## 6. Timing summary

Cost of building the dataset, per time step.

In [7]:
timing_rows = []
for result in generation_results:
    train_t = result['df_unique_train']['Processing time (s)']
    val_t   = result['df_unique_val']['Processing time (s)']
    timing_rows.append({
                           'Time (years)':   result['time_step'],
                           'n_train':        len(train_t),
                           'Train total (s)': train_t.sum(),
                           'Train mean (ms)': train_t.mean() * 1e3,
                           'n_val':          len(val_t),
                           'Val total (s)':  val_t.sum(),
                       })

emulator_timing = pd.DataFrame(timing_rows)

with open(f'{n_latent_samples}_emulator_timing_durability.pkl', 'wb') as f:
    dill.dump(emulator_timing, f)

train_total = emulator_timing['Train total (s)'].sum()
val_total   = emulator_timing['Val total (s)'].sum()

print(f"Train split - emulator g-value dataset generation time: {train_total:.1f} s")
print(f"Val split   - emulator g-value dataset generation time: {val_total:.1f} s")
print(f"Total emulator g-value dataset generation time (train + val): {train_total + val_total:.1f} s")
emulator_timing

Train split - emulator g-value dataset generation time: 151.2 s
Val split   - emulator g-value dataset generation time: 39.1 s
Total emulator g-value dataset generation time (train + val): 190.3 s


,Time (years),n_train,Train total (s),Train mean (ms),n_val,Val total (s)
0,0.0,200,28.875005,144.375024,50,7.408156
1,25.0,200,20.703904,103.519520,50,8.285589
2,50.0,200,36.241682,181.208410,50,10.785147
3,75.0,200,41.834757,209.173786,50,7.442250
4,100.0,200,23.529564,117.647821,50,5.157887


## 7. Unique dataset statistics

Concatenate the `dataset_unique` (train + val, all time steps) frames already held in
`generation_results` and describe the columns, including the fitted lambdas.

In [8]:
dataset_unique_all = pd.concat(
    [
        result[f'df_unique_{split}'].assign(split=split)
        for result in generation_results
        for split in ('train', 'val')
    ],
    ignore_index=True,
)

print(f"Combined unique dataset: {len(dataset_unique_all)} rows "
      f"({len(generation_results)} time steps x train/val splits)")
dataset_unique_all.describe()

Combined unique dataset: 1250 rows (5 time steps x train/val splits)


,fck,rh,cov,lambda 1,lambda 2,lambda 3,lambda 4,Processing time (s)
count,1250.000000,1250.000000,1250.000000,1250.000000,1250.000000,1250.000000,1250.000000,1250.000000
mean,34.169370,48.828777,37.203266,23.776423,1.695277,0.131748,0.142494,0.152211
std,8.526121,17.384004,12.074924,16.884527,0.685665,0.021241,0.020693,0.079082
min,20.035727,20.000270,15.174042,-36.552389,0.526451,0.057795,0.073898,0.088631
25%,27.157093,33.183072,27.641827,12.233020,1.264400,0.117488,0.128379,0.101974
50%,32.759317,48.635428,37.475050,24.906033,1.512255,0.131776,0.142852,0.121079
75%,41.723909,63.128855,47.112468,36.365075,1.935786,0.145811,0.156496,0.170675
max,49.799710,79.777076,59.816767,59.794389,4.691767,0.201402,0.206824,1.303674
